<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Classify_747.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget http://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/archives/fgvc-aircraft-2013b.tar.gz
!tar -xzf fgvc-aircraft-2013b.tar.gz

--2025-12-16 16:40:47--  http://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/archives/fgvc-aircraft-2013b.tar.gz
Resolving www.robots.ox.ac.uk (www.robots.ox.ac.uk)... 129.67.94.2
Connecting to www.robots.ox.ac.uk (www.robots.ox.ac.uk)|129.67.94.2|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/archives/fgvc-aircraft-2013b.tar.gz [following]
--2025-12-16 16:40:48--  https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/archives/fgvc-aircraft-2013b.tar.gz
Connecting to www.robots.ox.ac.uk (www.robots.ox.ac.uk)|129.67.94.2|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2753340328 (2.6G) [application/x-gzip]
Saving to: ‘fgvc-aircraft-2013b.tar.gz’

fgvc-aircraft-2013b 100%[===================>]   2.56G  7.45MB/s    in 3m 23s  

2025-12-16 16:44:11 (13.0 MB/s) - ‘fgvc-aircraft-2013b.tar.gz’ saved [2753340328/2753340328]



In [2]:
!pip install -q torch torchvision tqdm

In [3]:
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class AircraftDataset(Dataset):
    def __init__(self, split_file, image_dir, class_to_idx, transform=None):
        self.samples = []
        with open(split_file) as f:
            for line in f:
                img, label = line.strip().split(" ", 1)
                self.samples.append((img, label))
        self.image_dir = image_dir
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, label = self.samples[idx]
        img_path = os.path.join(self.image_dir, img_name + ".jpg")
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.class_to_idx[label]

In [4]:
# Read all labels
labels = set()
for split in ["train", "val"]:
    with open(f"fgvc-aircraft-2013b/data/images_variant_{split}.txt") as f:
        for line in f:
            labels.add(line.strip().split(" ", 1)[1])

class_names = sorted(labels)
class_to_idx = {c: i for i, c in enumerate(class_names)}

NUM_CLASSES = len(class_names)
print("Number of aircraft classes:", NUM_CLASSES)

Number of aircraft classes: 100


In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
import torch
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.resnet50(pretrained=True)
model.fc = torch.nn.Linear(2048, NUM_CLASSES)
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 61.4MB/s]


In [7]:
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

train_ds = AircraftDataset(
    "fgvc-aircraft-2013b/data/images_variant_train.txt",
    "fgvc-aircraft-2013b/data/images",
    class_to_idx,
    transform
)

val_ds = AircraftDataset(
    "fgvc-aircraft-2013b/data/images_variant_val.txt",
    "fgvc-aircraft-2013b/data/images",
    class_to_idx,
    transform
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

In [8]:
from tqdm import tqdm

EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for imgs, labels in tqdm(train_loader):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.3f}")

100%|██████████| 105/105 [00:50<00:00,  2.06it/s]


Epoch 1/20 | Loss: 369.010


100%|██████████| 105/105 [00:47<00:00,  2.23it/s]


Epoch 2/20 | Loss: 207.672


100%|██████████| 105/105 [00:47<00:00,  2.19it/s]


...

Epoch 18/20 | Loss: 14.017


100%|██████████| 105/105 [00:47<00:00,  2.20it/s]


Epoch 19/20 | Loss: 12.096


100%|██████████| 105/105 [00:49<00:00,  2.13it/s]

Epoch 20/20 | Loss: 10.953


In [9]:
torch.save(model.state_dict(), "aircraft_classifier_resnet50.pt")

In [11]:
from google.colab import drive
drive.mount("/content/drive")

!cp aircraft_classifier_resnet50.pt /content/drive/MyDrive/

Mounted at /content/drive
